# Lab Work - 4.4

# Q1. Cost-Complexity Pruning

## Theory

For each internal node:

α_eff(t) = ( RSS(t) − RSS(T_t) ) / ( |T_t| − 1 )

where:
- RSS(t) = RSS of node t
- RSS(T_t) = RSS of subtree rooted at t
- |T_t| = number of leaves in subtree

The node with smallest α_eff is called the weakest link and is pruned first.

### Pruning Sequence

T0 → T1 → T2 → ... → Root

Each pruning step removes the weakest subtree and replaces it with a leaf.

### Selecting Optimal α

Use cross-validation to estimate test RSS for every pruned tree.

Choose the α producing the lowest cross-validation RSS.

# Q2. Code Implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.tree import export_text

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.metrics import mean_squared_error

In [ ]:
# Dataset

X = np.array([[1],[2],[3],[4],[5],[6]])
y = np.array([10,20,25,28,40,45])

print('X =')
print(X)

print('\ny =')
print(y)

In [ ]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print('Training Samples:', len(X_train))
print('Testing Samples :', len(X_test))

In [ ]:
# Fully Grown Tree

full_tree = DecisionTreeRegressor(
    max_depth=None,
    random_state=42
)

full_tree.fit(X_train,y_train)

train_pred = full_tree.predict(X_train)
test_pred = full_tree.predict(X_test)

train_mse = mean_squared_error(y_train,train_pred)
test_mse = mean_squared_error(y_test,test_pred)

print('Train MSE:',train_mse)
print('Test MSE :',test_mse)

In [ ]:
# Different Depths

depths = [1,2,3,4,None]

results = []

for depth in depths:

    model = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train,y_train)

    train_mse = mean_squared_error(
        y_train,
        model.predict(X_train)
    )

    test_mse = mean_squared_error(
        y_test,
        model.predict(X_test)
    )

    results.append([depth,train_mse,test_mse])

results_df = pd.DataFrame(
    results,
    columns=['Max Depth','Train MSE','Test MSE']
)

results_df

In [ ]:
# Cross Validation

cv_results = []

for depth in depths:

    model = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42
    )

scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='neg_mean_squared_error'
    )

    mse = -scores.mean()

    cv_results.append([depth,mse])

cv_df = pd.DataFrame(
    cv_results,
    columns=['Depth','CV MSE']
)

cv_df

In [ ]:
# Best Depth

best_depth = cv_df.loc[
    cv_df['CV MSE'].idxmin(),
    'Depth'
]

print('Optimal Depth =',best_depth)

In [ ]:
# Final Model

final_tree = DecisionTreeRegressor(
    max_depth=None if pd.isna(best_depth) else best_depth,
    random_state=42
)

final_tree.fit(X,y)

print('Feature Importance')
print(final_tree.feature_importances_)

In [ ]:
# Tree Structure

tree_text = export_text(
    final_tree,
    feature_names=['x']
)

print(tree_text)

# Q4. Visualizations

In [ ]:
# Full Tree

plt.figure(figsize=(10,6))
plot_tree(full_tree,filled=True)
plt.title('Fully Grown Tree')
plt.show()

In [ ]:
# Pruned Tree

plt.figure(figsize=(10,6))
plot_tree(final_tree,filled=True)
plt.title('Pruned / Optimal Tree')
plt.show()

In [ ]:
# Prediction Curves

x_plot = np.linspace(1,6,200).reshape(-1,1)

plt.figure(figsize=(10,6))

plt.scatter(X,y,s=80,label='Actual Data')

plt.plot(
    x_plot,
    full_tree.predict(x_plot),
    label='Full Tree'
)

plt.plot(
    x_plot,
    final_tree.predict(x_plot),
    label='Pruned Tree'
)

plt.title('Prediction Comparison')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Train vs Test MSE

plt.figure(figsize=(8,5))

plt.plot(results_df['Max Depth'].astype(str),
         results_df['Train MSE'],
         marker='o',
         label='Train MSE')

plt.plot(results_df['Max Depth'].astype(str),
         results_df['Test MSE'],
         marker='o',
         label='Test MSE')

plt.title('Train vs Test Error')
plt.xlabel('Max Depth')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# CV Error Curve

plt.figure(figsize=(8,5))

plt.plot(
    cv_df['Depth'].astype(str),
    cv_df['CV MSE'],
    marker='o'
)

plt.title('Cross Validation Error')
plt.xlabel('Max Depth')
plt.ylabel('CV MSE')
plt.grid(True)
plt.show()

# Q4. Deep Intuition

## Q4.1 Hypothesis Space

A regression tree represents piecewise constant functions.

Unlike linear regression, it can model nonlinear relationships by recursively partitioning the feature space.

## Q4.2 Cost-Complexity Pruning

Cost-complexity pruning balances goodness of fit and model complexity.

Large α values encourage smaller trees.

Small α values allow deeper trees.

## Q4.3 Train MSE = 0 and Test MSE = 42

Diagnosis:
- Severe overfitting

Remedies:
1. Reduce max_depth
2. Increase minimum samples per leaf
4. Apply cost-complexity pruning

## Q4.4 Feature Importance

Feature importance measures how much a feature reduces RSS across all splits.

A larger value indicates a stronger contribution to prediction quality.

# Conclusion

- Deep trees can perfectly fit training data.
- Pruning improves generalization.
- Cross-validation helps choose the optimal tree size.
- Feature importance reveals which variables drive predictions.